In [6]:
from pathlib import Path
import json
import pandas as pd

# Automatically resolve project root whether running from root or notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_dir = PROJECT_ROOT / "data" / "raw"
processed_dir = PROJECT_ROOT / "data" / "processed"
interim_dir = PROJECT_ROOT / "data" / "interim"

df_train = pd.read_json(raw_dir / "train.jsonl", lines=True)
df_shared_task = pd.read_json(raw_dir / "shared_task_dev.jsonl", lines=True)

In [7]:
def cal_descriptive_stats(df: pd.DataFrame, name: str, output_dir=interim_dir) -> dict:
    """Calculate descriptive statistics for a given DataFrame and save to JSON."""
    # Convert any column containing lists to strings so nunique() and describe() can hash them
    df_hashable = df.copy()
    for col in df_hashable.columns:
        if df_hashable[col].apply(lambda x: isinstance(x, list)).any():
            df_hashable[col] = df_hashable[col].astype(str)

    # Replace NaN with None so output is valid JSON
    summary_df = df_hashable.describe(include="all")
    summary_clean = summary_df.where(pd.notna(summary_df), None).to_dict()

    stats = {
        "dataset_name": name,
        "num_rows": int(len(df)),
        "num_columns": int(len(df.columns)),
        "columns": df.columns.tolist(),
        "missing_values": df.isna().sum().to_dict(),
        "unique_values": df_hashable.nunique().to_dict(),
        "summary_statistics": summary_clean,
    }

    # Save to JSON
    filename = f"{name.lower().replace(' ', '_')}_stats.json"
    output_path = output_dir / filename
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(stats, f, indent=4, default=str)

    print(f"Saved descriptive stats for '{name}' -> {output_path}")
    return stats


train_stats = cal_descriptive_stats(df_train, "Training Data")
dev_stats = cal_descriptive_stats(df_shared_task, "Shared Task Data")

Saved descriptive stats for 'Training Data' -> c:\Users\Clint Muzanenhamo\ClaimCheck-AI\data\interim\training_data_stats.json
Saved descriptive stats for 'Shared Task Data' -> c:\Users\Clint Muzanenhamo\ClaimCheck-AI\data\interim\shared_task_data_stats.json


In [8]:


# Investigate target distribution across both splits (counts + percentages)
target_distribution = pd.DataFrame({
    "train_count": df_train["label"].value_counts(),
    "train_pct": (df_train["label"].value_counts(normalize=True) * 100),
    "shared_dev_count": df_shared_task["label"].value_counts(),
    "shared_dev_pct": (df_shared_task["label"].value_counts(normalize=True) * 100),
})

print(target_distribution)

# Save cleanly to JSON (orient="index" keeps label names as top-level keys)
target_distribution.to_json(
    interim_dir / "target_distribution.json",
    orient="index",
    indent=4,
)

                 train_count  train_pct  shared_dev_count  shared_dev_pct
label                                                                    
NOT ENOUGH INFO        35639  24.502747              6666       33.333333
REFUTES                29775  20.471093              6666       33.333333
SUPPORTS               80035  55.026160              6666       33.333333


In [9]:
plot = target_distribution.plot(
    kind="bar",
    stacked=True
)
plot.set_xlabel("Label")
plot.set_ylabel("Count")
plot.set_title("Target Distribution")
plot.legend(title="Dataset")        
plot.figure.savefig(interim_dir / "target_distribution.png", bbox_inches="tight")
print(f"Saved target distribution plot -> {interim_dir / 'target_distribution.png'}")


AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:


def audit_repeated_claims(df: pd.DataFrame, name: str) -> pd.DataFrame:
    print(f"=== {name} Repeated Claims Audit ===")
    
    # 1. Isolate all occurrences of repeated claims using duplicated(keep=False)
    dup_df = df[df.duplicated(subset=["claim"], keep=False)].copy()
    
    # Cast unhashable 'evidence' list to string so nunique() can compare annotations
    dup_df["evidence_str"] = dup_df["evidence"].astype(str)
    
    # 2. Group by 'claim' and count distinct labels, evidence annotations, and IDs
    summary = (
        dup_df.groupby("claim")
        .agg(
            occurrences=("id", "count"),
            unique_labels=("label", "nunique"),
            unique_evidence=("evidence_str", "nunique"),
            unique_ids=("id", "nunique"),
        )
        .reset_index()
    )
    
    print(f"Total rows involved in duplicate claims : {len(dup_df)}")
    print(f"Unique claims that are repeated        : {len(summary)}")
    print(f"  - With IDENTICAL labels              : {(summary['unique_labels'] == 1).sum()}")
    print(f"  - With DIFFERENT labels              : {(summary['unique_labels'] > 1).sum()}")
    print(f"  - With DIFFERENT evidence annotations: {(summary['unique_evidence'] > 1).sum()}")
    print(f"  - With DIFFERENT IDs                 : {(summary['unique_ids'] > 1).sum()}\n")
    
    return summary

# Run on both splits without removing any records
train_dup_summary = audit_repeated_claims(df_train, "Training Data")
dev_dup_summary = audit_repeated_claims(df_shared_task, "Shared Task Dev")

train_dup_summary.to_csv(interim_dir / "train_repeated_claims_summary.csv", index=False)
dev_dup_summary.to_csv(interim_dir / "dev_repeated_claims_summary.csv", index=False)

# Inspect examples of repeated claims with different labels (if any)
display(train_dup_summary[train_dup_summary["unique_labels"] > 1].head(10))

=== Training Data Repeated Claims Audit ===
Total rows involved in duplicate claims : 15798
Unique claims that are repeated        : 5894
  - With IDENTICAL labels              : 5141
  - With DIFFERENT labels              : 753
  - With DIFFERENT evidence annotations: 5894
  - With DIFFERENT IDs                 : 5894

=== Shared Task Dev Repeated Claims Audit ===
Total rows involved in duplicate claims : 736
Unique claims that are repeated        : 335
  - With IDENTICAL labels              : 281
  - With DIFFERENT labels              : 54
  - With DIFFERENT evidence annotations: 335
  - With DIFFERENT IDs                 : 335



,claim,occurrences,unique_labels,unique_evidence,unique_ids
10,21 Jump Street was released by a production co...,2,2,2,2
11,21 Jump Street was released in the United States.,2,2,2,2
16,24 is a historical fiction film.,2,2,2,2
18,24 is a thriller television show.,2,2,2,2
29,50 First Dates is a horror film.,2,2,2,2
46,A Nightmare on Elm Street stars Rooney Mara.,5,3,5,5
47,A Nightmare on Elm Street stars at least one a...,4,2,4,4
51,A Perfect Circle is a Canadian supergroup.,2,2,2,2
52,A Perfect Circle is a country supergroup.,2,2,2,2
61,A Song of Ice and Fire is a movie.,2,2,2,2


In [ ]:
# Investigate whether the same claim appears with different labels
df_train_claim_label = df_train.groupby("claim")["label"].nunique().reset_index(name="unique_labels")

conflicting_claims = df_train_claim_label[df_train_claim_label["unique_labels"] > 1]
print(f"Number of claims with multiple labels: {len(conflicting_claims)}")

# Inspect the actual rows for any conflicting claims
if not conflicting_claims.empty:
    df_train[df_train["claim"].isin(conflicting_claims["claim"])].sort_values("claim")[
        ["id", "claim", "label", "evidence"]
    ].head(10)

Number of claims with multiple labels: 753


In [ ]:
# investigate evidence structure and distribution

evidence_structure = df_train["evidence"].apply(lambda x: [len(group) for group in x] if isinstance(x, list) else []).tolist()
print(f"Evidence structure (number of groups per claim): {evidence_structure[:10]}")

Evidence structure (number of groups per claim): [[2], [1, 1], [1], [1], [1], [2], [1], [1], [1, 1], [1, 1]]


In [ ]:
evi_struct = type(df_train["evidence"].iloc[0])
print(f"Type of 'evidence' column: {evi_struct}")

Type of 'evidence' column: <class 'list'>


In [ ]:
evi_struct2 =  df_train.loc[110, "evidence"]

print(f"Example of 'evidence' structure for first claim: {evi_struct2}")

Example of 'evidence' structure for first claim: [[[172887, None, None, None]]]


In [ ]:
def check_split_overlap(df_train: pd.DataFrame, df_dev: pd.DataFrame) -> None:
    """Check independently for ID overlap and claim-text leakage across splits."""
    # 1. Exact ID overlap
    id_overlap = set(df_train["id"]).intersection(set(df_dev["id"]))
    
    # 2. Claim text overlap (true data leakage risk, regardless of ID)
    claim_overlap = set(df_train["claim"]).intersection(set(df_dev["claim"]))
    
    print(f"Overlapping IDs between Train and Dev    : {len(id_overlap)}")
    print(f"Overlapping Claims between Train and Dev : {len(claim_overlap)}")
    
    if claim_overlap:
        leaked_rows = df_dev[df_dev["claim"].isin(claim_overlap)][["id", "label", "claim"]].sort_values("claim")
        display(leaked_rows.head(10))

check_split_overlap(df_train, df_shared_task)

Overlapping IDs between Train and Dev    : 0
Overlapping Claims between Train and Dev : 3


,id,label,claim
17816,36399,SUPPORTS,Floyd Mayweather Jr. is a professional boxer.
8058,198218,SUPPORTS,Saturn is a planet in the Solar System.
10688,154383,SUPPORTS,Warren Beatty co-wrote Reds.


In [ ]:


#  Identify the 3 overlapping claim strings
overlapping_claims = set(df_train["claim"]).intersection(set(df_shared_task["claim"]))

#  Extract matching rows from both splits side-by-side
train_leaks = df_train[df_train["claim"].isin(overlapping_claims)][["id", "claim", "label", "evidence"]].assign(split="train")
dev_leaks = df_shared_task[df_shared_task["claim"].isin(overlapping_claims)][["id", "claim", "label", "evidence"]].assign(split="shared_task_dev")

leak_comparison = pd.concat([train_leaks, dev_leaks]).sort_values(["claim", "split"])
display(leak_comparison)

,id,claim,label,evidence,split
17816,36399,Floyd Mayweather Jr. is a professional boxer.,SUPPORTS,"[[[105200, 118615, Floyd_Mayweather_Jr., 0]], ...",shared_task_dev
45757,57807,Floyd Mayweather Jr. is a professional boxer.,SUPPORTS,"[[[74147, 85375, Floyd_Mayweather_Jr., 0]], [[...",train
8058,198218,Saturn is a planet in the Solar System.,SUPPORTS,"[[[233056, 236058, Saturn, 0]]]",shared_task_dev
100294,142165,Saturn is a planet in the Solar System.,SUPPORTS,"[[[165224, 179348, Solar_System, 7], [165224, ...",train
10688,154383,Warren Beatty co-wrote Reds.,SUPPORTS,"[[[293532, 286037, Warren_Beatty, 2]], [[29740...",shared_task_dev
10795,216752,Warren Beatty co-wrote Reds.,SUPPORTS,"[[[257528, 256524, Reds_-LRB-film-RRB-, 0]], [...",train


In [ ]:
# Investigate how to obtain the count for each label

label_counts = df_train["label"].value_counts().reset_index()
label_counts.columns = ["label", "count"]
label_counts

,label,count
0,SUPPORTS,80035
1,NOT ENOUGH INFO,35639
2,REFUTES,29775


In [ ]:
# Investigate how to obtain the total number of records

total_records = len(df_train)
label_counts["proportion"] = label_counts["count"] / total_records
total_records

145449

In [ ]:
from statsmodels.stats.proportion import proportion_confint




# 3. Calculate 95% Wilson confidence intervals and assign to columns
ci_low, ci_upper = proportion_confint(
    count=label_counts["count"],
    nobs=total_records,
    alpha=0.05,
    method="wilson",
)
label_counts["ci_low"] = ci_low
label_counts["ci_upper"] = ci_upper

print(f"Wilson confidence intervals calculated for each label at 95% confidence level with {total_records} total records.")

# Display final table
label_counts[["label", "count", "proportion", "ci_low", "ci_upper"]]

Wilson confidence intervals calculated for each label at 95% confidence level with 145449 total records.


,label,count,proportion,ci_low,ci_upper
0,SUPPORTS,80035,0.550262,0.547704,0.552817
1,NOT ENOUGH INFO,35639,0.245027,0.242824,0.247245
2,REFUTES,29775,0.204711,0.202645,0.206792


In [ ]:
# contingency table for label across (train vs shared_task_dev)
import pandas as pd

# Combine both splits with a 'split' column so every row is counted
combined_labels = pd.concat([
    pd.DataFrame({"split": "train", "label": df_train["label"]}),
    pd.DataFrame({"split": "shared_task_dev", "label": df_shared_task["label"]}),
], ignore_index=True)

# Build the true contingency table across splits
contingency_table = pd.crosstab(
    combined_labels["split"],
    combined_labels["label"],
    margins=True,
    margins_name="Total",
)
contingency_table


label,NOT ENOUGH INFO,REFUTES,SUPPORTS,Total
split,,,,
shared_task_dev,6666,6666,6666,19998
train,35639,29775,80035,145449
Total,42305,36441,86701,165447


In [ ]:

# Run a chi-square test.
from scipy.stats import chi2_contingency


chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table.iloc[:-1, :-1])  # Exclude the 'Total' row and column

print(f"Chi-square statistic: {chi2_stat}")
print(f"P-value: {p_value:3e}")
print(f"Degrees of freedom: {dof}")
print(f"Expected frequencies:\n{expected}")

Chi-square statistic: 3435.375105625399
P-value: 0.000000e+00
Degrees of freedom: 2
Expected frequencies:
[[ 5113.51302834  4404.71642278 10479.77054888]
 [37191.48697166 32036.28357722 76221.22945112]]


In [ ]:
# Calculate Cramér's V to measure the strength of the association

import numpy as np

# Calculate Cramér's V
n = contingency_table.iloc[:-1, :-1].values.sum()  # Total number of observations (excluding the 'Total' row and column)
min_dim = min(contingency_table.shape) - 1  # Minimum of (rows - 1, columns - 1)
cramers_v = np.sqrt(chi2_stat / (n * min_dim))
print(f"Cramér's V: {cramers_v}")


Cramér's V: 0.10189259804282169


In [10]:
import pandas as pd

def inspect_evidence(ev_list):

    result = {
        "has_annotation_structure": isinstance(ev_list, list),
        "evidence_groups": 0,
        "annotation_items": 0,
        "usable_evidence_items": 0,
        "incomplete_evidence_items": 0,
        "missing_annotator_ids": 0,
        "missing_evidence_ids": 0,
        "missing_page_titles": 0,
        "missing_sentence_ids": 0,
        "malformed_items": 0
    }

    if not isinstance(ev_list, list):
        return result

    result["evidence_groups"] = len(ev_list)

    for group in ev_list:

        if not isinstance(group, list):
            result["malformed_items"] += 1
            continue

        for item in group:

            result["annotation_items"] += 1

            if not isinstance(item, (list, tuple)) or len(item) < 4:
                result["malformed_items"] += 1
                result["incomplete_evidence_items"] += 1
                continue

            annotator_id, evidence_id, page, sentence_id = item[:4]

            if annotator_id is None:
                result["missing_annotator_ids"] += 1

            if evidence_id is None:
                result["missing_evidence_ids"] += 1

            if page is None:
                result["missing_page_titles"] += 1

            if sentence_id is None:
                result["missing_sentence_ids"] += 1

            if page is not None and sentence_id is not None:
                result["usable_evidence_items"] += 1
            else:
                result["incomplete_evidence_items"] += 1

    return result

In [11]:
train_evidence_audit = pd.DataFrame(
    df_train["evidence"].apply(inspect_evidence).tolist()
)

dev_evidence_audit = pd.DataFrame(
    df_shared_task["evidence"].apply(inspect_evidence).tolist()
)

train_evidence_audit.insert(0, "label", df_train["label"])

train_evidence_summary = (
    train_evidence_audit
    .groupby("label")
    .sum(numeric_only=True)
)

dev_evidence_audit.insert(0, "label", df_shared_task["label"])

dev_evidence_summary = (
    dev_evidence_audit
    .groupby("label")
    .sum(numeric_only=True)
)
print("TRAINING EVIDENCE ASSESSMENT")
display(train_evidence_summary)

print("DEVELOPMENT EVIDENCE ASSESSMENT")
display(dev_evidence_summary)

TRAINING EVIDENCE ASSESSMENT


,has_annotation_structure,evidence_groups,annotation_items,usable_evidence_items,incomplete_evidence_items,missing_annotator_ids,missing_evidence_ids,missing_page_titles,missing_sentence_ids,malformed_items
label,,,,,,,,,,
NOT ENOUGH INFO,35639,35639,47609,0,47609,0,47609,47609,47609,0
REFUTES,29775,60227,70066,70066,0,0,0,0,0,0
SUPPORTS,80035,161249,193756,193756,0,0,0,0,0,0


DEVELOPMENT EVIDENCE ASSESSMENT


,has_annotation_structure,evidence_groups,annotation_items,usable_evidence_items,incomplete_evidence_items,missing_annotator_ids,missing_evidence_ids,missing_page_titles,missing_sentence_ids,malformed_items
label,,,,,,,,,,
NOT ENOUGH INFO,6666,6666,8941,0,8941,0,8941,8941,8941,0
REFUTES,6666,12227,14017,14017,0,0,0,0,0,0
SUPPORTS,6666,12498,14608,14608,0,0,0,0,0,0


In [12]:
### 5.1 Baseline Data Integrity Checks
def validate_dataset(df, name):

    results = {
        "dataset": name,
        "total_records": len(df),
        "duplicate_ids": df["id"].duplicated().sum(),
        "missing_ids": df["id"].isna().sum(),
        "missing_claims": df["claim"].isna().sum(),
        "empty_claims": (
            df["claim"]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        ),
        "missing_labels": df["label"].isna().sum(),
        "invalid_labels": (
            ~df["label"].isin([
                "SUPPORTS",
                "REFUTES",
                "NOT ENOUGH INFO"
            ])
        ).sum(),
        "missing_evidence_columns": df["evidence"].isna().sum()
    }

    return results


baseline_results = pd.DataFrame([
    validate_dataset(df_train, "Training"),
    validate_dataset(df_shared_task, "Development")
])

display(baseline_results)

,dataset,total_records,duplicate_ids,missing_ids,missing_claims,empty_claims,missing_labels,invalid_labels,missing_evidence_columns
0,Training,145449,0,0,0,0,0,0,0
1,Development,19998,0,0,0,0,0,0,0
